# The Knowledge Agent (RAG)

**What you will build:** the milestone of Block 1 — an agent that answers questions from *your* documents by retrieving the relevant passages first. This is **RAG** (retrieval-augmented generation), and it's the cure for two problems from 0.0: the model's knowledge is frozen and private, and you can't fit everything in the context window. Maps to chapter 09 of the n8n course.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/17_knowledge_agent_rag.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"   # verified against pydantic-ai 2.2 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)
!pip install -q sentence-transformers

## The idea in one picture

Don't paste all your documents into the prompt (they won't fit, and it's wasteful). Instead: **embed** each chunk into a vector, **store** the vectors, and at question time **retrieve** the few chunks closest to the question and put only those in front of the model.

```
  documents ─embed─▶ vectors ──store──┐
                                       │   question ─embed─▶ vector
                                       ▼                        │
                              similarity search  ◀──────────────┘
                                       │
                              top-k relevant chunks ─▶ into the prompt ─▶ answer
```

We use `sentence-transformers` for embeddings — it runs locally on Colab's CPU, needs **no API key**, and won't rot.

## Step 1 — Embed and store your documents

A tiny knowledge base. In a real project these would be chunks of your PDFs, docs, or a wiki.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # 384-dim, CPU-friendly, no API key

documents = [
    "ACME return policy: items can be returned within 30 days with a receipt.",
    "ACME ships to the EU and the UK. Standard delivery takes 3 to 5 business days.",
    "ACME Pro plan costs 20 euros per month and includes priority support.",
    "ACME support hours are Monday to Friday, 9am to 6pm CET.",
    "Warranty: ACME hardware is covered for 2 years against manufacturing defects.",
]
doc_vectors = embedder.encode(documents, normalize_embeddings=True)   # (N, 384) unit vectors
print("stored", len(documents), "documents as", doc_vectors.shape, "vectors")

## Step 2 — Retrieve the most relevant chunks

Cosine similarity is just a dot product on normalized vectors. No database needed for a small set.

In [ ]:
def retrieve(query, k=2):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = doc_vectors @ q                    # cosine similarity to every doc
    top = np.argsort(-scores)[:k]
    return [documents[i] for i in top]

print(retrieve("how long do I have to send something back?"))   # finds the return policy
print(retrieve("what does the pro plan cost?"))

Notice the query said "send something back," not "return" — but retrieval still found the return policy. That's the point of embeddings: they match on **meaning**, not keywords.

## Step 3 — Give retrieval to the agent as a tool

The clean way to do RAG with an agent: expose retrieval as a **tool**. Now the agent decides *when* it needs to look something up — and can search more than once. This is *agentic* RAG, and it's just a tool (1.2) whose body happens to be a vector search.

In [ ]:
rag_agent = Agent(
    model,
    system_prompt=(
        "You are ACME's support assistant. Use the search_docs tool to find relevant policy before answering, "
        "and answer ONLY from what it returns. If the docs don't cover it, say you don't know."
    ),
)

@rag_agent.tool_plain
def search_docs(query: str) -> str:
    """Search ACME's policy documents for passages relevant to the query."""
    return "\n".join(retrieve(query, k=2))

result = await rag_agent.run("I bought a laptop 40 days ago and it's faulty. Am I covered?")
print(result.output)

The agent searched the docs, found the 30-day return window *and* the 2-year warranty, and reasoned about which applies. It grounded its answer in your data — and because you told it to say "I don't know" when the docs are silent, it won't invent policy. RAG is not a special framework feature; it's retrieval wired in as a tool.

::::{dropdown} 🔍 RAG vs memory vs long context
:color: info

```
Long context   paste everything      simple, but expensive and hits the window limit
Memory (0.6)   keep recent turns     good for conversation, not for a big knowledge base
RAG            fetch only what's     scales to millions of chunks; the model sees only
               relevant, on demand   the few passages it needs this turn
```

For a handful of docs, plain retrieval like this is plenty. Past ~50k chunks, swap the numpy search for `faiss-cpu` (`faiss.IndexFlatIP(384)`) — same cosine math, built for scale. In Block 2 you'll see RAG again with a grading loop (*agentic RAG*, 2.7).
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add your own docs.** Replace `documents` with a few facts about a topic you know, re-embed, and ask about them.
2. **Watch grounding work.** Ask something the docs don't cover ("do you sell cars?") and confirm it says it doesn't know instead of inventing.
3. **Tune k.** Return `k=1` vs `k=4` chunks and see how retrieval breadth changes the answer's completeness.
::::

::::{dropdown} 🔧 Common issues (and fixes)
:color: secondary

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| **Says "I don't know" when the answer IS in the docs** | Retrieval missed the chunk (`k` too small, chunks too big) | Raise `k`; split documents into smaller chunks |
| **Retrieves irrelevant passages** | Query and docs phrased very differently | Try a stronger embedding model; add a query-rewrite step (see 2.7) |
| **Answers from its own knowledge, not the docs** | System prompt not strict enough | Tell it to answer ONLY from retrieved passages, else say "I don't know" |
| **Slow on many documents** | Linear numpy scan over all vectors | Switch to `faiss-cpu` (`IndexFlatIP`) for large collections |
::::

You can now build clean, typed, tool-using agents with memory, guardrails, evals, and retrieval — the core toolkit for most real applications.

**What's next:** two more Block 1 skills — in **18** you get tools from an **MCP server** (capabilities you didn't write), and in **19** you learn to **debug** an agent when it misbehaves.